# Latih YOLO bola + landmark (Colab / Kaggle)

Kelas: `0 bola, 1 tiang_gawang, 2 titik_penalti, 3 titik_tengah, 4 sudut_lapangan`
(sama dengan `dataset_gen` dan jenis landmark lokalisasi GoNasional).

Langkah:
1. Di laptop: buat dataset simulasi lalu zip.
   ```bash
   ros2 run aroc_sim_bridge dataset_gen --out ~/datasets/sim_multi --count 5000 --seed 1
   cd ~/datasets && zip -qr sim_multi.zip sim_multi
   ```
   Dataset foto asli tim juga di-zip (struktur `images/{train,val}`, `labels/{train,val}`).
2. Upload zip ke Google Drive (folder `yolo_aroc`), atau sebagai Kaggle Dataset.
3. Runtime -> GPU (T4), jalankan sel dari atas.

Hasil akhir: `best.pt` dan folder `best_openvino_model/` (input tetap `IMGSZ`). Salin folder
OpenVINO ke `GoNasional/model/` di robot dan jalankan `main_vision` dengan `-p imgsz:=IMGSZ`.
`main_vision` hanya mengambil kelas bernama `bola`, jadi kelas landmark tidak mengganggu.

In [ ]:
%pip install -q ultralytics==8.4.60 openvino
import os, glob, subprocess
IMGSZ = 480          # harus sama dengan -p imgsz di main_vision; 480 cukup untuk frame 450x337
EPOCHS = 80
MODEL = 'yolo11n.pt' # n = paling ringan untuk CPU robot

## Data
Isi `ZIPS` dengan zip dataset. Format tiap entri: `(path_zip, MAP)`. `MAP` memetakan indeks
kelas dataset itu ke daftar di atas, mis. dataset tim `GK, Robot, bola` -> `'2=0'`
(hanya bola yang dipakai). `None` = indeks sudah sesuai (dataset `dataset_gen`).

In [ ]:
IN_COLAB = 'google.colab' in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/yolo_aroc'
    WORK = '/content/work'
else:  # Kaggle: dataset di /kaggle/input/<nama> (sudah diekstrak, tulis path folder di ZIPS)
    BASE = '/kaggle/input'
    WORK = '/kaggle/working'

ZIPS = [
    (f'{BASE}/sim_multi.zip', None),
    # (f'{BASE}/gkeep_sementara.zip', '2=0'),
]
os.makedirs(f'{WORK}/raw', exist_ok=True)
for path, _ in ZIPS:
    if os.path.isdir(path):   # Kaggle mengekstrak zip yang di-upload sebagai Dataset
        subprocess.run(['cp', '-r', path, f'{WORK}/raw/'], check=True)
    else:
        subprocess.run(['unzip', '-qo', path, '-d', f'{WORK}/raw'], check=True)
print(os.listdir(f'{WORK}/raw'))

In [ ]:
# merge_datasets.py dari repo op3-krsbi-sim
!wget -q -O merge_datasets.py https://raw.githubusercontent.com/afifidf/op3-krsbi-sim/main/training/merge_datasets.py
args = []
for path, remap in ZIPS:
    name = os.path.splitext(os.path.basename(path))[0]
    args += ['--src', f'{WORK}/raw/{name}' + (f':{remap}' if remap else '')]
subprocess.run(['python', 'merge_datasets.py', '--out', f'{WORK}/data'] + args, check=True)

## Latih

In [ ]:
from ultralytics import YOLO
model = YOLO(MODEL)
model.train(data=f'{WORK}/data/data.yaml', imgsz=IMGSZ, epochs=EPOCHS, batch=32,
            project=f'{WORK}/runs', name='aroc', exist_ok=True,
            # augmentasi warna lebih kuat: menutup sebagian beda render Webots vs kamera asli
            hsv_h=0.02, hsv_s=0.6, hsv_v=0.5, degrees=5, mosaic=1.0, close_mosaic=10)

In [ ]:
best = f'{WORK}/runs/aroc/weights/best.pt'
metrics = YOLO(best).val(data=f'{WORK}/data/data.yaml', imgsz=IMGSZ)
for i, name in metrics.names.items():
    print(f'{name:16s} mAP50 {metrics.box.maps[i]:.3f}')

## Ekspor OpenVINO (input tetap) dan simpan

In [ ]:
YOLO(best).export(format='openvino', imgsz=IMGSZ, half=False)
out = f'{BASE}/hasil' if IN_COLAB else f'{WORK}/hasil'
os.makedirs(out, exist_ok=True)
subprocess.run(['cp', '-r', best, best.replace('.pt', '_openvino_model'), out], check=True)
print('Tersimpan di', out)